# 17 — Two new detectors trained in parallel: Faster R-CNN v2 + FCOS at 960 px (Kaggle T4×2)

**Goal:** add two strong, *different* models to the ensemble (notebook 16b) without YOLO and without GitHub downloads. Both are torchvision models with COCO weights from `download.pytorch.org`.

| model | family | COCO box AP (torchvision) | why |
|---|---|---|---|
| Faster R-CNN ResNet-50 FPN **v2** | two-stage, anchor-based | 46.7 (v1 used so far: 37.0) | improved recipe (BN in FPN/heads, conv box head) |
| **FCOS** ResNet-50 FPN | one-stage, anchor-free | 39.2 | per-pixel assignment, different errors from the anchor models |

**What changes compared with the earlier runs (all motivated by the EDA):**
- **Resolution:** frames are upsampled to 960 px short side (1707×960) instead of RetinaNet/Faster R-CNN's 1280×720 and RT-DETR's 640×640: 24 % of objects are below 32×32 px.
- **Multi-scale training** (short side 832–1088, picked randomly per batch), horizontal flip.
- **Smaller anchors** for Faster R-CNN v2 (16–256 px instead of 32–512).
- **Ignored regions are greyed out in training images** (≈21 ignored boxes per frame, 75 % of frames). Before, they were simply dropped, so unlabelled objects inside them were learned as background. COCOeval already ignores detections inside them.
- Training: all 56 train videos, every 4th frame, SGD + warm-up + cosine, AMP, frozen BatchNorm statistics (batch 2). **Number of epochs is fitted to a time budget** after measuring the first 100 iterations, so the run always finishes inside a Kaggle session.

**How to run (Kaggle):** attach the competition dataset · Accelerator **GPU T4 ×2** · Internet **on** · **Save Version → Save & Run All**. Each GPU trains one model; with `TIME_BUDGET_H = 6` the whole notebook takes ≈7 h. Then download `hr_models_bundle.zip` from Output and upload it to `MyDrive/UCU_AI_HW2/`; notebook 16b picks it up automatically.

On Colab (one GPU) the two models run one after another with half the budget each, and everything is saved to `MyDrive/UCU_AI_HW2/hr_models/` (rerun after a disconnect to resume).

Holdout hygiene: the 7 validation videos are never trained on; the *last* epoch is exported (no checkpoint selection on the holdout used by 16b). The per-epoch "quick" holdout mAP (every 9th frame) is only a training curve.

In [1]:
%pip -q install pycocotools

Note: you may need to restart the kernel to use updated packages.


## 1 — Environment and configuration

In [2]:
import os, sys, json, time, shutil, zipfile, subprocess, io, contextlib, gzip, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch

TIME_BUDGET_H = 6.0      # wall-clock hours for training + prediction, per GPU
RESERVE_H     = 0.7      # part of the budget kept for prediction caches (holdout + test, with and without flip)
COMMON = dict(train_sizes=[832, 896, 1024, 1088, 960],   # random short side in training; the LAST one is used at inference
              max_size=1824, frame_step=4, batch=2, epochs=8, warmup_iters=500, seed=42, mask_ignored=True)
MODELS = {
    'frcnn2': dict(arch='frcnn2', lr=0.005, anchor_sizes=[16, 32, 64, 128, 256]),
    'fcos':   dict(arch='fcos',   lr=0.0025),
}

# Kaggle images also contain the google.colab package, so check Kaggle FIRST.
IN_KAGGLE = Path('/kaggle/input').is_dir() and Path('/kaggle/working').is_dir()
IN_COLAB = False
if not IN_KAGGLE:
    try:
        from google.colab import drive; IN_COLAB = True
    except ImportError:
        pass
assert torch.cuda.is_available(), 'Enable a GPU accelerator.'
N_GPU = torch.cuda.device_count()
if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/UCU_AI_HW2'); OUT = DRIVE / 'hr_models'
elif IN_KAGGLE:
    DRIVE = None; OUT = Path('/kaggle/working/hr_models')
else:
    DRIVE = None; OUT = Path('hr_models').resolve()
CACHE = OUT / 'cache'; CACHE.mkdir(parents=True, exist_ok=True)
PARALLEL = N_GPU >= len(MODELS)
budget = TIME_BUDGET_H if PARALLEL else TIME_BUDGET_H / len(MODELS)
print(f'Kaggle={IN_KAGGLE} Colab={IN_COLAB} GPUs={N_GPU} ({torch.cuda.get_device_name(0)}) | '
      f'{"parallel" if PARALLEL else "sequential"}, {budget:.1f} h per model | output {OUT}')

Kaggle=True Colab=False GPUs=2 (Tesla T4) | parallel, 6.0 h per model | output /kaggle/working/hr_models


In [3]:
def find_data():
    """Kaggle: attached competition input. Colab: extract visdrone_competition.zip once to local disk."""
    def find(base):
        return [p.parent.parent for p in Path(base).rglob('train/annotations_coco.json')
                if (p.parent.parent / 'val/images').is_dir() and (p.parent.parent / 'test/images').is_dir()]
    if IN_KAGGLE:
        found = find('/kaggle/input')
    else:
        base = Path('/content/visdrone'); base.mkdir(exist_ok=True)
        found = find(base)
        if not found:
            print('Extracting dataset to local disk (one time, ~5 min)...')
            with zipfile.ZipFile(DRIVE / 'visdrone_competition.zip') as z:
                z.extractall(base)
            found = find(base)
    assert len(found) == 1, found
    return found[0]

DATA = find_data()
# Download COCO weights once in the main process (torchvision cache) so both workers start immediately.
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights, FCOS_ResNet50_FPN_Weights
for w in (FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1, FCOS_ResNet50_FPN_Weights.COCO_V1):
    for attempt in range(5):
        try:
            w.get_state_dict(progress=False); break
        except Exception as e:
            print('weight download retry', attempt + 1, e); time.sleep(15 * (attempt + 1))
print('Data:', DATA, '| COCO weights cached')

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
Downloading: "https://download.pytorch.org/models/fcos_resnet50_fpn_coco-99b0c9b7.pth" to /root/.cache/torch/hub/checkpoints/fcos_resnet50_fpn_coco-99b0c9b7.pth
Data: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026 | COCO weights cached


## 2 — Training script

One script trains one model on one GPU (selected with `CUDA_VISIBLE_DEVICES`) and then writes prediction caches in the exact format of notebooks 14/15b/16b. Running it as a separate process lets the two models train simultaneously and keeps this notebook's output short (full logs: `hr_models/<name>_log.txt`).

In [4]:
SCRIPT_TEXT = r'''"""Fine-tune ONE torchvision detector on VisDrone train videos, then cache holdout/test predictions.
Usage: CUDA_VISIBLE_DEVICES=<gpu> python train_det.py <config.json>     (written by notebook 17)"""
import os, sys, io, re, json, gzip, math, time, random, hashlib, contextlib
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, torch, cv2
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ---- torchvision detectors trained in notebook 17 (also used by 16b to rebuild them) ----
import torch.nn as nn
from torchvision.models.detection import (fasterrcnn_resnet50_fpn_v2, fcos_resnet50_fpn,
                                          FasterRCNN_ResNet50_FPN_V2_Weights, FCOS_ResNet50_FPN_Weights)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator

HR_LABEL_SHIFT = {'frcnn2': -1, 'fcos': 0}   # model label -> competition class 0..7

def build_hr_model(cfg, pretrained=False, min_size=None):
    """cfg['arch'] in {'frcnn2','fcos'}. pretrained=True loads COCO weights from download.pytorch.org."""
    sizes = tuple(cfg['train_sizes']) if min_size is None else (int(min_size),)   # eval uses the LAST size
    common = dict(min_size=sizes, max_size=cfg['max_size'])
    if cfg['arch'] == 'frcnn2':
        m = fasterrcnn_resnet50_fpn_v2(
            weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1 if pretrained else None, weights_backbone=None,
            rpn_pre_nms_top_n_train=2000, rpn_post_nms_top_n_train=2000,
            rpn_pre_nms_top_n_test=2000, rpn_post_nms_top_n_test=1000,
            box_detections_per_img=300, box_score_thresh=0.001, box_nms_thresh=0.5,
            box_batch_size_per_image=512, **common)
        # anchors halved (16..256 px instead of 32..512): 24% of objects are < 32x32 px
        m.rpn.anchor_generator = AnchorGenerator(sizes=tuple((s,) for s in cfg['anchor_sizes']),
                                                 aspect_ratios=((0.5, 1.0, 2.0),) * 5)
        m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 9)
    elif cfg['arch'] == 'fcos':
        m = fcos_resnet50_fpn(weights=FCOS_ResNet50_FPN_Weights.COCO_V1 if pretrained else None,
                              weights_backbone=None, score_thresh=0.001, nms_thresh=0.6,
                              detections_per_img=300, topk_candidates=1000, **common)
        h = m.head.classification_head          # keep the pretrained conv tower, replace only the logits
        h.cls_logits = nn.Conv2d(h.cls_logits.in_channels, h.num_anchors * 8, 3, padding=1)
        nn.init.normal_(h.cls_logits.weight, std=0.01)
        nn.init.constant_(h.cls_logits.bias, -math.log((1 - 0.01) / 0.01))
        h.num_classes = 8
    else:
        raise ValueError(cfg['arch'])
    return m

def load_hr_model(path, min_size=None, device='cuda'):
    payload = torch.load(path, map_location='cpu', weights_only=True)
    m = build_hr_model(payload['config'], pretrained=False, min_size=min_size)
    missing, unexpected = m.load_state_dict(payload['model'], strict=False)
    # FrozenBatchNorm (pretrained build) has no num_batches_tracked buffer; everything else must match
    assert not unexpected and all(k.endswith('num_batches_tracked') for k in missing), (missing[:5], unexpected[:5])
    return m.eval().to(device), payload['config']

# ---- raw-prediction caches: {image_key: {'size':[w,h], 'detections':[[cls0..7, score, x1,y1,x2,y2], ...]}} ----
MIN_CONF, MAX_RAW_DETS = 0.001, 300
_FP = {}
def fingerprint(path):
    path = Path(path)
    if path not in _FP:
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            while block := f.read(8 * 1024 * 1024):
                h.update(block)
        _FP[path] = h.hexdigest()[:12]
    return _FP[path]

def _clean(boxes, classes, scores, w, h, flip):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4).copy()
    if flip:  # undo horizontal flip
        boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, w)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, h)
    classes = np.asarray(classes, dtype=int); scores = np.asarray(scores, dtype=float)
    ok = (classes >= 0) & (classes < 8) & (scores >= MIN_CONF) & (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    order = np.argsort(-scores[ok], kind='stable')[:MAX_RAW_DETS]
    return [[int(c), float(s), *map(float, b)] for c, s, b in
            zip(classes[ok][order], scores[ok][order], boxes[ok][order])]

def predict_ultralytics(model, paths, imgsz, flip=False, batch=8, half=True, desc=''):
    """YOLO or RT-DETR (Ultralytics). flip=True runs on the mirrored frame and mirrors boxes back."""
    items, res = list(paths.items()), {}
    for s in tqdm(range(0, len(items), batch), desc=desc, leave=False):
        chunk = items[s:s + batch]
        if flip:
            src = [np.ascontiguousarray(cv2.imread(str(p))[:, ::-1]) for _, p in chunk]
        else:
            src = [str(p) for _, p in chunk]
        preds = model.predict(source=src, imgsz=imgsz, conf=MIN_CONF, iou=0.7, max_det=MAX_RAW_DETS,
                              half=half, device=0, batch=len(chunk), verbose=False, save=False, augment=False)
        assert len(preds) == len(chunk)
        for (key, _), r in zip(chunk, preds):
            h, w = r.orig_shape
            if r.boxes is None or len(r.boxes) == 0:
                dets = []
            else:
                dets = _clean(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(),
                              r.boxes.conf.cpu().numpy(), w, h, flip)
            res[str(key)] = {'size': [int(w), int(h)], 'detections': dets}
    return res

def save_cache(res, dest):
    tmp = Path(str(dest) + '.partial')
    with gzip.open(tmp, 'wt') as f:
        json.dump(res, f)
    tmp.replace(dest)

def load_cache(path):
    with gzip.open(path, 'rt') as f:
        return json.load(f)


C = json.loads(Path(sys.argv[1]).read_text())
NAME, DATA = C['name'], Path(C['data'])
OUT = Path(C['out']); CACHE = Path(C['cache']); CACHE.mkdir(parents=True, exist_ok=True)
LAST, FINAL, DONE = OUT / f'{NAME}_last.pt', OUT / f'{NAME}_final.pt', OUT / f'{NAME}_done.json'
SHIFT = HR_LABEL_SHIFT[C['arch']]

def log(*a):
    s = time.strftime('%H:%M:%S ') + ' '.join(map(str, a))
    print(s, flush=True)
    with open(OUT / f'{NAME}_log.txt', 'a') as f:
        f.write(s + '\n')

def status(**kw):
    (OUT / f'{NAME}_status.json').write_text(json.dumps(dict(kw, name=NAME, time=time.strftime('%H:%M'))))

random.seed(C['seed']); np.random.seed(C['seed']); torch.manual_seed(C['seed'])
dev = torch.device('cuda')

# ------------------------------- data -------------------------------
train = json.loads((DATA / 'train/annotations_coco.json').read_text())
VAL_COCO = json.loads((DATA / 'val/annotations_coco.json').read_text())
by_img = defaultdict(list)
for a in train['annotations']:
    by_img[a['image_id']].append(a)
clips = defaultdict(list)
for info in sorted(train['images'], key=lambda i: i['file_name']):
    clips[Path(info['file_name']).parent.name].append(info)
assert len(clips) == 56
items = []
for clip in sorted(clips):
    for k, info in enumerate(clips[clip]):
        if k % C['frame_step']:
            continue
        W, H = info['width'], info['height']; boxes, labels, ign = [], [], []
        for a in by_img[info['id']]:
            x, y, w, h = map(float, a['bbox'])
            x1, y1, x2, y2 = max(0., x), max(0., y), min(W, x + w), min(H, y + h)
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            if int(a.get('iscrowd', 0)) == 1:
                ign.append([x1, y1, x2, y2])
            elif 1 <= int(a['category_id']) <= 8:
                boxes.append([x1, y1, x2, y2]); labels.append(int(a['category_id']) - 1 - SHIFT)
        if boxes:
            items.append((info['file_name'], boxes, labels, ign))
del train, by_img
log(f'{NAME}: {len(items)} training frames (step {C["frame_step"]}, 56 videos)')

MEAN = np.array([124, 116, 104], np.uint8)
def _sl(b):
    return slice(int(b[1]), int(math.ceil(b[3]))), slice(int(b[0]), int(math.ceil(b[2])))

class Frames(Dataset):
    def __len__(self):
        return len(items)
    def __getitem__(self, i):
        fn, boxes, labels, ign = items[i]
        arr = np.array(Image.open(DATA / 'train' / fn).convert('RGB'))
        if C['mask_ignored'] and ign:
            # grey out ignored regions so un-annotated objects are not learned as background,
            # but restore pixels of annotated objects that overlap them
            orig = arr.copy()
            for b in ign:
                arr[_sl(b)] = MEAN
            for b in boxes:
                arr[_sl(b)] = orig[_sl(b)]
        bx = torch.tensor(boxes, dtype=torch.float32)
        if random.random() < 0.5:
            arr = arr[:, ::-1]; W = arr.shape[1]
            bx = torch.stack([W - bx[:, 2], bx[:, 1], W - bx[:, 0], bx[:, 3]], 1)
        x = torch.from_numpy(np.ascontiguousarray(arr)).permute(2, 0, 1).float().div_(255)
        return x, {'boxes': bx, 'labels': torch.tensor(labels, dtype=torch.int64)}

def collate(b):
    return tuple(zip(*b))

loader = DataLoader(Frames(), batch_size=C['batch'], shuffle=True, num_workers=C['workers'], collate_fn=collate,
                    pin_memory=True, drop_last=True, persistent_workers=C['workers'] > 0)
IPE = len(loader)

# holdout ids (same definition as 14/15b/16b) + quick per-epoch check on every 9th frame
meta = pd.DataFrame([{'id': int(i['id']), 'file_name': i['file_name'], 'clip': Path(i['file_name']).parent.name}
                     for i in VAL_COCO['images']]).sort_values(['clip', 'file_name']).reset_index(drop=True)
meta['n'] = meta.groupby('clip').cumcount()
VAL = meta[meta.n % 3 == 0]; QUICK = meta[meta.n % 9 == 0]
assert len(VAL) == 951
gt_path = OUT / 'val_coco.json'
if not gt_path.is_file():
    gt_path.write_text(json.dumps(VAL_COCO))
with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(str(gt_path))

def predict(model, paths, flip, desc):
    res = {}
    model.eval()
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for key, p in tqdm(paths.items(), desc=desc, mininterval=60):
            with Image.open(p) as im:
                w, h = im.size
                x = torch.from_numpy(np.array(im.convert('RGB'))).permute(2, 0, 1).to(dev).float().div_(255)
            if flip:
                x = x.flip(-1)
            d = model([x])[0]
            res[str(key)] = {'size': [w, h], 'detections': _clean(d['boxes'].float().cpu().numpy(),
                             d['labels'].cpu().numpy() + SHIFT, d['scores'].float().cpu().numpy(), w, h, flip)}
    return res

def coco_map(res, ids):
    dets = [{'image_id': int(i), 'category_id': int(c) + 1, 'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': s}
            for i in ids for c, s, x1, y1, x2, y2 in res[str(i)]['detections']]
    if not dets:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        e = COCOeval(GT, GT.loadRes(dets), 'bbox')
        e.params.imgIds = [int(i) for i in ids]; e.params.catIds = list(range(1, 9)); e.params.maxDets = [1, 10, 100]
        e.evaluate(); e.accumulate(); e.summarize()
    return float(e.stats[0])

def val_path(fn):
    return DATA / 'val' / fn

# ------------------------------- model / optimiser -------------------------------
model = build_hr_model(C, pretrained=True).to(dev)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=C['lr'], momentum=0.9, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda')
S = dict(epoch=0, it=0, total=IPE * C['epochs'], planned=False, history=[])
if LAST.is_file() and not DONE.is_file():
    ck = torch.load(LAST, map_location='cpu', weights_only=True)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler']); S = ck['state']
    log(f'resumed after epoch {S["epoch"]}')

def lr_at(it):
    w = C['warmup_iters']
    if it < w:
        return C['lr'] * (0.001 + 0.999 * it / w)
    p = min(1.0, (it - w) / max(1, S['total'] - w))
    return C['lr'] * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * p)))   # cosine to 5% of base LR

def set_train():
    model.train()
    for m in model.modules():          # batch of 2 is too small for BatchNorm statistics: keep COCO stats
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

t0 = time.time(); budget = C['budget_h'] * 3600; reserve = C['reserve_h'] * 3600; ep_time = None
while not DONE.is_file() and S['epoch'] * IPE < S['total']:
    if ep_time and time.time() - t0 + ep_time > budget - reserve:
        log('time budget reached, stopping early'); break
    ep = S['epoch'] + 1; set_train(); te = time.time(); run = n = 0
    for imgs, tgts in loader:
        imgs = [i.to(dev, non_blocking=True) for i in imgs]
        tgts = [{k: v.to(dev) for k, v in t.items()} for t in tgts]
        for g in opt.param_groups:
            g['lr'] = lr_at(S['it'])
        with torch.autocast('cuda', dtype=torch.float16):
            loss = sum(model(imgs, tgts).values())
        opt.zero_grad(set_to_none=True)
        S['it'] += 1
        if not torch.isfinite(loss):
            log('non-finite loss, batch skipped'); continue
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 10.0)
        scaler.step(opt); scaler.update()
        run += loss.item(); n += 1
        if not S['planned'] and n == 100:     # fit the number of epochs to the time budget
            sec = (time.time() - te) / 100
            fit = int((budget - reserve) / (sec * IPE))
            S['total'] = max(1, min(C['epochs'], fit)) * IPE; S['planned'] = True
            log(f'{sec:.2f} s/iter x {IPE} iters/epoch -> {S["total"] // IPE} epochs fit the budget')
        if n % 250 == 0:
            status(epoch=ep, iter=n, iters=IPE, loss=round(run / n, 4), planned_epochs=S['total'] // IPE)
    ep_time = time.time() - te
    quick = coco_map(predict(model, {str(r.id): val_path(r.file_name) for r in QUICK.itertuples()}, False, 'quick'),
                     QUICK.id.tolist())
    S['epoch'] = ep
    S['history'].append(dict(epoch=ep, loss=round(run / max(n, 1), 4), quick_holdout_mAP=round(quick, 4),
                             minutes=round(ep_time / 60, 1), lr=lr_at(S['it'])))
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'state': S},
               str(LAST) + '.tmp'); os.replace(str(LAST) + '.tmp', LAST)
    pd.DataFrame(S['history']).to_csv(OUT / f'{NAME}_history.csv', index=False)
    log(f'epoch {ep}/{S["total"] // IPE}: loss {run / max(n, 1):.4f}, quick holdout mAP {quick:.4f}, {ep_time / 60:.0f} min')
    status(epoch=ep, iter=IPE, iters=IPE, loss=round(run / max(n, 1), 4), quick_mAP=round(quick, 4),
           planned_epochs=S['total'] // IPE)

# ------------------------------- export + prediction caches -------------------------------
if not FINAL.is_file():
    torch.save({'model': model.state_dict(), 'config': C}, FINAL)   # last epoch (not selected on the holdout)
model, _ = load_hr_model(FINAL)
fp = fingerprint(FINAL)
vp = {str(r.id): val_path(r.file_name) for r in VAL.itertuples()}
troot = DATA / 'test' / 'images'
tp = dict(sorted({f'{p.parent.name}_{p.stem}': p for p in troot.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')}.items()))
assert len(tp) == 3811
for split, paths in (('val_step3', vp), ('test_all', tp)):
    for flip in (False, True):
        label = NAME + ('_flip' if flip else '')
        dest = CACHE / f'{split}_{label}_{fp}.json.gz'
        if not dest.is_file():
            status(stage=f'predicting {split} flip={flip}')
            save_cache(predict(model, paths, flip, f'{label} {split}'), dest)
            log('cached', dest.name)
DONE.write_text(json.dumps({'epochs': S['epoch'], 'history': S['history']}))
status(stage='done', epochs=S['epoch'])
log('DONE')
'''

SCRIPT = Path('train_det.py').resolve()
SCRIPT.write_text(SCRIPT_TEXT)
import ast; ast.parse(SCRIPT_TEXT)
print('wrote', SCRIPT, len(SCRIPT_TEXT.splitlines()), 'lines')

wrote /kaggle/working/train_det.py 320 lines


## 3 — Train both models (≈ `TIME_BUDGET_H` hours) and write prediction caches

In [5]:
def launch(name, gpu):
    cfg = dict(COMMON, **MODELS[name], name=name, data=str(DATA), out=str(OUT), cache=str(CACHE),
               budget_h=budget, reserve_h=RESERVE_H,
               workers=max(1, min(4, (os.cpu_count() or 2) // (len(MODELS) if PARALLEL else 1))))
    path = OUT / f'{name}_config.json'; path.write_text(json.dumps(cfg, indent=1))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen([sys.executable, str(SCRIPT), str(path)], env=env,
                            stdout=open(OUT / f'{name}_stdout.txt', 'a'), stderr=subprocess.STDOUT)

def monitor(procs):
    seen = {}          # print a status line only when a model finishes an epoch or changes stage
    while any(p.poll() is None for p in procs.values()):
        time.sleep(120)
        for name in procs:
            f = OUT / f'{name}_status.json'
            if not f.is_file(): continue
            st = json.loads(f.read_text()); key = (st.get('epoch'), st.get('iter') == st.get('iters'), st.get('stage'))
            if key != seen.get(name) and (st.get('iter') in (None, st.get('iters')) or name not in seen):
                seen[name] = key; print(st)
    for name, p in procs.items():
        if p.returncode != 0:
            print(f'--- {name} FAILED, last log lines ---'); print((OUT / f'{name}_stdout.txt').read_text()[-3000:])
    return all(p.returncode == 0 for p in procs.values())

todo = [n for n in MODELS if not (OUT / f'{n}_done.json').is_file()]
print('already finished:', [n for n in MODELS if n not in todo], '| to run:', todo)
ok = True
if PARALLEL:
    ok = monitor({n: launch(n, i) for i, n in enumerate(todo)}) if todo else True
else:
    for n in todo:
        ok = monitor({n: launch(n, 0)}) and ok
assert ok, 'A training process failed (see log above). Rerun this cell to resume from the last epoch.'
for n in MODELS:
    h = OUT / f'{n}_history.csv'
    if h.is_file():
        print(n); display(pd.read_csv(h))

already finished: [] | to run: ['frcnn2', 'fcos']
{'epoch': 1, 'iter': 250, 'iters': 3035, 'loss': 1.3787, 'planned_epochs': 8, 'name': 'frcnn2', 'time': '19:03'}
{'epoch': 1, 'iter': 250, 'iters': 3035, 'loss': 1.4686, 'planned_epochs': 8, 'name': 'fcos', 'time': '19:03'}
{'epoch': 1, 'iter': 3035, 'iters': 3035, 'loss': 0.9041, 'quick_mAP': 0.2504, 'planned_epochs': 8, 'name': 'frcnn2', 'time': '19:31'}
{'epoch': 1, 'iter': 3035, 'iters': 3035, 'loss': 1.145, 'quick_mAP': 0.2033, 'planned_epochs': 8, 'name': 'fcos', 'time': '19:32'}
{'epoch': 2, 'iter': 3035, 'iters': 3035, 'loss': 0.7312, 'quick_mAP': 0.2565, 'planned_epochs': 8, 'name': 'frcnn2', 'time': '20:03'}
{'epoch': 2, 'iter': 3035, 'iters': 3035, 'loss': 1.0535, 'quick_mAP': 0.2029, 'planned_epochs': 8, 'name': 'fcos', 'time': '20:04'}
{'epoch': 3, 'iter': 3035, 'iters': 3035, 'loss': 0.671, 'quick_mAP': 0.2549, 'planned_epochs': 8, 'name': 'frcnn2', 'time': '20:34'}
{'epoch': 3, 'iter': 3035, 'iters': 3035, 'loss': 1.0221,

,epoch,loss,quick_holdout_mAP,minutes,lr
0,1,0.9041,0.2504,29.9,0.004868
1,2,0.7312,0.2565,30.0,0.004385
2,3,0.6710,0.2549,29.8,0.003624
3,4,0.6236,0.2639,30.0,0.002703
4,5,0.5867,0.2462,29.9,0.001771
5,6,0.5549,0.2572,29.9,0.000974
6,7,0.5309,0.2578,29.8,0.000438
7,8,0.5167,0.2573,29.9,0.000250


fcos


,epoch,loss,quick_holdout_mAP,minutes,lr
0,1,1.1450,0.2033,30.7,0.002434
1,2,1.0535,0.2029,30.7,0.002193
2,3,1.0221,0.2098,30.6,0.001812
3,4,1.0014,0.2076,30.7,0.001352
4,5,0.9844,0.2034,30.8,0.000885
5,6,0.9726,0.1984,30.8,0.000487
6,7,0.9643,0.1997,30.7,0.000219
7,8,0.9601,0.2000,30.7,0.000125


## 4 — Holdout COCOeval (951 frames, 7 videos, original `iscrowd` regions) and single-model submissions

In [6]:
# ---- raw-prediction caches: {image_key: {'size':[w,h], 'detections':[[cls0..7, score, x1,y1,x2,y2], ...]}} ----
MIN_CONF, MAX_RAW_DETS = 0.001, 300
_FP = {}
def fingerprint(path):
    path = Path(path)
    if path not in _FP:
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            while block := f.read(8 * 1024 * 1024):
                h.update(block)
        _FP[path] = h.hexdigest()[:12]
    return _FP[path]

def _clean(boxes, classes, scores, w, h, flip):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4).copy()
    if flip:  # undo horizontal flip
        boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, w)
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, h)
    classes = np.asarray(classes, dtype=int); scores = np.asarray(scores, dtype=float)
    ok = (classes >= 0) & (classes < 8) & (scores >= MIN_CONF) & (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    order = np.argsort(-scores[ok], kind='stable')[:MAX_RAW_DETS]
    return [[int(c), float(s), *map(float, b)] for c, s, b in
            zip(classes[ok][order], scores[ok][order], boxes[ok][order])]


def load_cache(path):
    with gzip.open(path, 'rt') as f:
        return json.load(f)

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(str(OUT / 'val_coco.json'))
meta = pd.DataFrame([{'id': i['id'], 'file_name': i['file_name'], 'clip': Path(i['file_name']).parent.name}
                     for i in GT.dataset['images']]).sort_values(['clip', 'file_name'])
VAL_IDS = meta[meta.groupby('clip').cumcount() % 3 == 0].id.astype(int).tolist()

def coco_eval(raw):
    dets = [{'image_id': int(i), 'category_id': int(c) + 1, 'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': s}
            for i in VAL_IDS for c, s, x1, y1, x2, y2 in raw[str(i)]['detections']]
    with contextlib.redirect_stdout(io.StringIO()):
        e = COCOeval(GT, GT.loadRes(dets), 'bbox')
        e.params.imgIds = VAL_IDS; e.params.catIds = list(range(1, 9)); e.params.maxDets = [1, 10, 100]
        e.evaluate(); e.accumulate(); e.summarize()
    s = e.stats
    return dict(mAP=s[0], AP50=s[1], AP75=s[2], AP_small=s[3], AP_medium=s[4], AP_large=s[5])

rows = []
for name in MODELS:
    fp = fingerprint(OUT / f'{name}_final.pt')
    for label in (name, name + '_flip'):
        rows.append({'source': label, **coco_eval(load_cache(CACHE / f'val_step3_{label}_{fp}.json.gz'))})
res = pd.DataFrame(rows).sort_values('mAP', ascending=False)
res.to_csv(OUT / 'holdout_coco_metrics.csv', index=False)
display(res.round(4))
print('Reference holdout mAP: RetinaNet final 0.165, RT-DETR x3 0.151, Faster R-CNN v1 (CV) ~0.167, 15b ensemble 0.209')

for name in MODELS:
    fp = fingerprint(OUT / f'{name}_final.pt')
    raw = load_cache(CACHE / f'test_all_{name}_{fp}.json.gz')
    rows = [{'image_id': k, 'PredictionString': ' '.join(
             f'{int(c)} {s:.6f} {x1:.2f} {y1:.2f} {x2 - x1:.2f} {y2 - y1:.2f}' for c, s, x1, y1, x2, y2 in raw[k]['detections'])}
            for k in sorted(raw)]
    sub = pd.DataFrame(rows); assert len(sub) == 3811
    sub.to_csv(OUT / f'submission_{name}.csv', index=False)
print('single-model submissions written to', OUT)

,source,mAP,AP50,AP75,AP_small,AP_medium,AP_large
1,frcnn2_flip,0.2545,0.5189,0.2190,0.1159,0.2483,0.4197
0,frcnn2,0.2542,0.5191,0.2260,0.1019,0.2465,0.4246
2,fcos,0.1972,0.3998,0.1674,0.0879,0.1847,0.3596
3,fcos_flip,0.1946,0.3941,0.1678,0.0927,0.1838,0.3597


Reference holdout mAP: RetinaNet final 0.165, RT-DETR x3 0.151, Faster R-CNN v1 (CV) ~0.167, 15b ensemble 0.209
single-model submissions written to /kaggle/working/hr_models


## 5 — Bundle for notebook 16b (Kaggle only; on Colab everything is already on Drive)

In [7]:
if IN_KAGGLE:
    bundle = Path('/kaggle/working/hr_models_bundle.zip')
    keep = [p for p in OUT.iterdir() if p.is_file() and (p.name.endswith(('_final.pt', '_history.csv', '_log.txt',
            '_config.json', '_done.json')) or p.name == 'holdout_coco_metrics.csv')] + sorted(CACHE.glob('*.json.gz'))
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_STORED) as z:
        for p in keep:
            z.write(p, arcname='hr_models/' + str(p.relative_to(OUT)))
    print(f'Download {bundle} ({bundle.stat().st_size / 1e6:.0f} MB) from Output -> upload to MyDrive/UCU_AI_HW2/')

Download /kaggle/working/hr_models_bundle.zip (404 MB) from Output -> upload to MyDrive/UCU_AI_HW2/
